# 🦶 Toe Be Sure — API Server
Downloads `best_dfu.pt` from GitHub and exposes it via ngrok.

**Run all 4 cells top to bottom. Cell 4 will print your `COLAB_URL`.**

Requirement: Free ngrok token from https://ngrok.com/signup

In [ ]:
# ── Cell 1: Install dependencies ─────────────────────────────────────────
!pip -q install torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip -q install flask flask-cors pyngrok git+https://github.com/jacobgil/pytorch-grad-cam
print("✅ Dependencies installed")

In [ ]:
# ── Cell 2: Download model from GitHub ───────────────────────────────────
import os

MODEL_PATH = '/content/best_dfu.pt'

if not os.path.exists(MODEL_PATH):
    print("Downloading model from GitHub...")
    !wget -q --show-progress -O /content/best_dfu.pt \
        "https://github.com/kharsha006/hackathon_dfu/raw/main/models/best_dfu.pt"
    print(f"✅ Model downloaded: {os.path.getsize(MODEL_PATH)/1e6:.1f} MB")
else:
    print(f"✅ Model already present: {os.path.getsize(MODEL_PATH)/1e6:.1f} MB")

In [ ]:
# ── Cell 3: Load ResNet18 model ───────────────────────────────────────────
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms

CLASS_NAMES = ["Abnormal(Ulcer)", "Normal(Healthy skin)"]
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, len(CLASS_NAMES))
model.load_state_dict(torch.load("/content/best_dfu.pt", map_location=device))
model.to(device)
model.eval()
print("✅ ResNet18 loaded")

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

In [ ]:
# ── Cell 4 (OPTIONAL): Evaluate model accuracy on your Drive dataset ──────
# Run this after Cell 3. Mount Drive first if not already mounted.
# from google.colab import drive; drive.mount('/content/drive')

import os, random, numpy as np
from torch.utils.data import DataLoader
from torchvision import datasets
from sklearn.metrics import (roc_auc_score, f1_score, accuracy_score,
                              confusion_matrix, classification_report)
import torch

# ── Point this to your unzipped dataset folder ────────────────────────────
def find_dataset(root):
    for dp, dns, _ in os.walk(root):
        if "Abnormal(Ulcer)" in dns and "Normal(Healthy skin)" in dns:
            return dp
    return None

DATA_DIR = find_dataset("/content/dfu_data")          # already unzipped?
if DATA_DIR is None:
    DATA_DIR = find_dataset("/content/drive/MyDrive/hackathon")  # from Drive

print("Dataset:", DATA_DIR)
if DATA_DIR is None:
    raise RuntimeError("Dataset not found. Unzip DFU.zip first (run Cell from training notebook).")

# ── Build a reproducible 15% test split ───────────────────────────────────
full = datasets.ImageFolder(DATA_DIR, transform=val_transforms)
idx  = list(range(len(full)))
random.seed(42); random.shuffle(idx)
test_idx = idx[int(0.85 * len(full)):]

class SubsetDS(torch.utils.data.Dataset):
    def __init__(self, ds, idxs): self.ds, self.idxs = ds, idxs
    def __len__(self): return len(self.idxs)
    def __getitem__(self, i): return self.ds[self.idxs[i]]

test_loader = DataLoader(SubsetDS(full, test_idx), batch_size=32,
                         shuffle=False, num_workers=2)

# ── Run inference ──────────────────────────────────────────────────────────
all_probs, all_preds, all_y = [], [], []
model.eval()
with torch.no_grad():
    for x, y in test_loader:
        x, y = x.to(device), y.to(device)
        probs = torch.softmax(model(x), dim=1)
        all_probs.append(probs[:, 0].cpu().numpy())
        all_preds.append(probs.argmax(1).cpu().numpy())
        all_y.append(y.cpu().numpy())

all_probs = np.concatenate(all_probs)
all_preds = np.concatenate(all_preds)
all_y     = np.concatenate(all_y)

binary_y     = (all_y == 0).astype(int)
binary_preds = (all_preds == 0).astype(int)
tn, fp, fn, tp = confusion_matrix(binary_y, binary_preds).ravel()

print("\n" + "=" * 52)
print("       ACCURACY REPORT — best_dfu.pt")
print("=" * 52)
print(f"  Test samples  : {len(all_y)}")
print(f"  Accuracy      : {accuracy_score(all_y, all_preds)*100:.2f}%")
print(f"  AUC-ROC       : {roc_auc_score(binary_y, all_probs):.4f}")
print(f"  F1 Score      : {f1_score(binary_y, binary_preds):.4f}")
print(f"  Sensitivity   : {tp/(tp+fn)*100:.2f}%  (ulcer correctly detected)")
print(f"  Specificity   : {tn/(tn+fp)*100:.2f}%  (healthy correctly rejected)")
print("=" * 52)
print(classification_report(all_y, all_preds, target_names=CLASS_NAMES))

In [ ]:
# ── Cell 4: Start Flask API + ngrok ──────────────────────────────────────
# 👉 Paste your authtoken from https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_AUTHTOKEN = '3AWz45FG18QLiyWc1cxno19XvSP_4Ls6W7v1Gw2ZdKdyoXC57'
NGROK_DOMAIN    = 'unillusive-unreally-sharolyn.ngrok-free.dev'

import io, base64, threading, numpy as np
from flask import Flask, request, jsonify
from flask_cors import CORS
from PIL import Image
import torch.nn.functional as F
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pyngrok import ngrok, conf

app = Flask(__name__)
CORS(app)

@app.route('/health', methods=['GET'])
def health():
    return jsonify({'status': 'ok', 'model': 'ResNet18 best_dfu.pt'})

@app.route('/analyze', methods=['POST'])
def analyze():
    if 'image' not in request.files:
        return jsonify({'error': 'No image field in request'}), 400

    file = request.files['image']
    img = Image.open(file.stream).convert('RGB')
    img_np = np.array(img.resize((224, 224))) / 255.0

    tensor = val_transforms(img).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(tensor)
        probs = torch.softmax(logits, dim=1)[0]

    ulcer_prob   = float(probs[0])
    healthy_prob = float(probs[1])
    confidence   = round(max(ulcer_prob, healthy_prob) * 100, 1)

    heatmap_b64 = None
    try:
        cam = GradCAM(model=model, target_layers=[model.layer4[-1]])
        grayscale_cam = cam(input_tensor=tensor, targets=[ClassifierOutputTarget(0)])[0]
        visualization = show_cam_on_image(img_np.astype(np.float32), grayscale_cam, use_rgb=True)
        buf = io.BytesIO()
        Image.fromarray(visualization).save(buf, format='PNG')
        heatmap_b64 = base64.b64encode(buf.getvalue()).decode('utf-8')
    except Exception as e:
        print(f'GradCAM skipped: {e}')

    return jsonify({
        'ulcer_probability':   round(ulcer_prob, 6),
        'healthy_probability': round(healthy_prob, 6),
        'prediction': CLASS_NAMES[0] if ulcer_prob > healthy_prob else CLASS_NAMES[1],
        'confidence': confidence,
        'heatmap': heatmap_b64
    })

# Start ngrok with static domain
ngrok.set_auth_token(NGROK_AUTHTOKEN)
tunnel = ngrok.connect(addr=5000, hostname=NGROK_DOMAIN)
public_url = f"https://{NGROK_DOMAIN}"

print('\n' + '='*60)
print('✅  TBS API is LIVE')
print(f'🌐  URL     : {public_url}')
print(f'🔗  Analyze : {public_url}/analyze')
print(f'💚  Health  : {public_url}/health')
print()
print('👉  Add this to Vercel Environment Variables:')
print(f'    COLAB_URL={public_url}')
print('='*60)
print('\n⚠️  Keep this cell running — closing stops the server.\n')

threading.Thread(target=lambda: app.run(port=5000, use_reloader=False)).start()